# SFT Training

This notebook trains Qwen3.5-9B-Base on our prepared Tulu 3 mixtures using Tinker. Run from the project root after completing `data_preparation.ipynb`.

## 1. Additional Environment Setup

Create a `.env` file in the project root containing `TINKER_API_KEY=your-api-key-here`. The training script loads it automatically. Do not commit `.env`.

## 2. Train a Model

Start with the balanced 5k dataset, one epoch, and the fixed validation set. `--no_publish` keeps the checkpoint unpublished during experiments. Training uses Tinker credits. Replace `smoke_test` with the training dataset. 

In [ ]:
!python train_sft.py --train_data data/mixtures/smoke_test.jsonl --val_data data/validation.jsonl --num_epochs 1 --batch_size 16 --lr 1e-4 --rank 32 --max_length 2048 --eval_every 100 --seed 42 --checkpoint_name smoke_test --output_dir outputs/smoke_test --no_publish

# !python train_sft.py --train_data data/mixtures/balanced_5k.jsonl --val_data data/validation.jsonl --num_epochs 1 --batch_size 16 --lr 1e-4 --rank 32 --max_length 2048 --eval_every 100 --seed 42 --checkpoint_name balanced_5k --output_dir outputs/balanced_5k --no_publish

## 3. View Training Results

The output directory contains `checkpoint_info.json` and the training/validation loss history. Use a different output directory and checkpoint name for each experiment.

In [2]:
import json
from pathlib import Path

info = json.loads(Path("outputs/balanced_5k/checkpoint_info.json").read_text())
print(json.dumps(info, indent=2))

{
  "checkpoint_path": "tinker://ba0bda34-6116-5379-ba1d-a7c76d8cbb5d:train:0/sampler_weights/balanced_5k",
  "training_state_path": "tinker://ba0bda34-6116-5379-ba1d-a7c76d8cbb5d:train:0/weights/balanced_5k",
  "checkpoint_name": "balanced_5k",
  "base_model": "Qwen/Qwen3.5-9B-Base",
  "renderer_name": "role_colon",
  "train_data": "data/mixtures/balanced_5k.jsonl",
  "val_data": "data/validation.jsonl",
  "training": {
    "num_epochs": 1,
    "num_steps": 313,
    "batch_size": 16,
    "learning_rate": 0.0001,
    "lora_rank": 32,
    "max_length": 2048,
    "eval_every": 100,
    "seed": 42,
    "beta1": 0.9,
    "beta2": 0.95,
    "eps": 1e-08
  },
  "published": false
}


## 4. Evaluate the Checkpoint

Run the provided evaluation script on the saved sampling checkpoint. Validation loss measures fit to reference responses; benchmark scores measure task performance.

In [3]:
checkpoint_path = info["checkpoint_path"]
base_model = info["base_model"]

!python -m evaluation.eval_all --checkpoint_path "{checkpoint_path}" --base_model "{base_model}"

INFO:__main__:Evaluating: tinker://ba0bda34-6116-5379-ba1d-a7c76d8cbb5d:train:0/sampler_weights/balanced_5k
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: Qwen/Qwen3.5-9B-Base | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session ab63adc9-7a55-5f51-aa14-9690d5ae898f
['  File "<frozen runpy>", line 198, in _run_module_as_main\n', '  File "<frozen runpy>", line 88, in _run_code\n', '  File "/Users/soonprinceton/Projects/Duke/cps375-575-mini-project/evaluation/eval_all.py", line 226, in <module>\n    main()\n', '  File "/Users/soonprinceton/Projects/Duke/cps375-575-mini-project/evaluation/eval_all.py", line 165, in main\n    metrics, task_results = asyncio.run(run_core(\n', '  File 

## 5. Command-Line Arguments

| Argument | Default / example | Description |
|---|---|---|
| `--train_data` | Required | Training JSONL path. |
| `--val_data` | `data/validation.jsonl` | Fixed validation JSONL path. |
| `--num_epochs` | 1 | Number of training epochs. |
| `--batch_size` | 16 (example) | Number of samples per batch. |
| `--lr` | 1e-4 | Learning rate. |
| `--rank` | 32 | LoRA rank. |
| `--max_length` | 2048 | Maximum sequence length in tokens. |
| `--eval_every` | 100 | Validation interval in optimizer steps. |
| `--save_every` | 0 | Save sampler and training checkpoints every N steps (0 disables periodic saves) |
| `--seed` | 42 | Training shuffle seed. |
| `--checkpoint_name` | `balanced_5k` (example) | Tinker checkpoint name. |
| `--output_dir` | `outputs/balanced_5k` (example) | Local experiment output directory. |
| `--no_publish` | False | Skip publishing the sampling checkpoint. |

For other mixtures, change `--train_data`, `--checkpoint_name`, and `--output_dir`. Keep the validation set fixed across experiments.

For the exact defaults supported by the implemented script, run the following command.

In [ ]:
!python train_sft.py --help

In [1]:
from dotenv import load_dotenv
load_dotenv()
!python -m evaluation.eval_all --checkpoint_path "Qwen/Qwen3.5-9B-Base" --base_model "Qwen/Qwen3.5-9B-Base"

INFO:__main__:Evaluating: Qwen/Qwen3.5-9B-Base
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: Qwen/Qwen3.5-9B-Base | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session b175249a-41d8-58dd-bcf8-4935a2cd8ab8
['  File "<frozen runpy>", line 198, in _run_module_as_main\n', '  File "<frozen runpy>", line 88, in _run_code\n', '  File "/Users/soonprinceton/Projects/Duke/cps375-575-mini-project/evaluation/eval_all.py", line 224, in <module>\n    main()\n', '  File "/Users/soonprinceton/Projects/Duke/cps375-575-mini-project/evaluation/eval_all.py", line 163, in main\n    metrics, task_results = asyncio.run(run_core(\n', '  File "/opt/homebrew/Caskroom/miniforge/base/envs/cs575-proj/lib/py